In [57]:
def RLE_step1(b):
    new_b = b""
    n = 1
    for i in range(len(b) - 1):
        if b[i] == b[i+1]:
            n += 1
        else:
            new_b += n.to_bytes(1, byteorder='big') + b[i:i+1]
            n = 1
    new_b += n.to_bytes(1, byteorder='big') + b[-1:]

    return new_b

def deRLE_step1(b):
    deB = b""
    for i in range(1, len(b), 2):
        deB += b[i:i+1]*b[i-1]
    
    return deB

b = b"\xCF\xCF\xCF\xCF\xCF\xFF"
print(b) # изначальная строка
new_b = RLE_step1(b)
print(new_b) # сжатая строка
deB = deRLE_step1(new_b)
print(deB) # разжатая строка

b'\xcf\xcf\xcf\xcf\xcf\xff'
b'\x05\xcf\x01\xff'
b'\xcf\xcf\xcf\xcf\xcf\xff'


In [132]:
def RLE_step2(b):
    new_b = b""
    n = 1
    i = 0
    while i < len(b) - 1:
        if b[i] == b[i+1]:
            n += 1
            i += 1
        else:
            if n > 1:
                new_b += n.to_bytes(1, byteorder='big') + b[i:i+1]
                i += 1
                n = 1
            else:
                buff = b""
                while i + 1 < len(b):
                    if b[i] == b[i+1]:
                        break
                    buff += b[i:i+1]
                    i += 1
                
                if i + 1 == len(b):
                    buff += b[i:i+1]
                    i += 1

                new_b += (len(buff) | 0x80).to_bytes(1, byteorder='big') + buff
                n = 1
    
    if i < len(b):
        if n > 1:
            new_b += n.to_bytes(1, byteorder='big') + b[-1:]
        else:
            new_b += ((1 | 0x80).to_bytes(1, byteorder='big') + b[-1:])

    return new_b

def deRLE_step2(b):
    deB = b""
    i = 0
    while i < len(b):
        if b[i] & 0x80:
            count = b[i] & 0x7F
            deB += b[i+1:i+1+count]
            i += count + 1
        else:
            deB += b[i+1:i+2]*b[i]
            i += 2
    
    return deB

b1 = b"\xCF\xCF\xCF\xCF\xCF\xFF"
print(b1) # изначальная строка
new_b1 = RLE_step2(b1)
print(new_b1) # сжатая строка
deB1 = deRLE_step2(new_b1)
print(deB1) # разжатая строка
print('\n')

b2 = b"\xCF\xCE\xCF\xCE\xCF" 
print(b2) # изначальная строка
new_b2 = RLE_step2(b2)
print(new_b2) # сжатая строка
deB2 = deRLE_step2(new_b2)
print(deB2) # разжатая строка
print('\n')

b3 = b"\xCF\xCE\xCF\xCE\xCF\xCF\xCF\xCF\xCF\xCF" 
print(b3) # изначальная строка
new_b3 = RLE_step2(b3)
print(new_b3) # сжатая строка
deB3 = deRLE_step2(new_b3)
print(deB3) # разжатая строка

b'\xcf\xcf\xcf\xcf\xcf\xff'
b'\x05\xcf\x81\xff'
b'\xcf\xcf\xcf\xcf\xcf\xff'


b'\xcf\xce\xcf\xce\xcf'
b'\x85\xcf\xce\xcf\xce\xcf'
b'\xcf\xce\xcf\xce\xcf'


b'\xcf\xce\xcf\xce\xcf\xcf\xcf\xcf\xcf\xcf'
b'\x84\xcf\xce\xcf\xce\x06\xcf'
b'\xcf\xce\xcf\xce\xcf\xcf\xcf\xcf\xcf\xcf'


In [23]:
def RLE_step3(b, Ms = 8, Mc = 8):

    new_b = b""
    n = 1
    i = 0
    symbol_size = Ms // 8
    control_size = Mc // 8
    while i < len(b) - symbol_size:
        bytes1 = b[i:i+symbol_size]
        bytes2 = b[i+symbol_size:i+2*symbol_size]
        if bytes1 == bytes2:
            n += 1
            i += symbol_size
        else:
            if n > 1:
                new_b += n.to_bytes(control_size, byteorder='big') + b[i:i+symbol_size]
                i += symbol_size
                n = 1
            else:
                buff = b""
                while i + symbol_size < len(b):
                    bytes1 = b[i:i+symbol_size]
                    bytes2 = b[i+symbol_size:i+2*symbol_size]
                    if bytes1 == bytes2:
                        break
                    buff += b[i:i+symbol_size]
                    i += symbol_size
                
                if i + symbol_size == len(b):
                    buff += b[i:i+symbol_size]
                    i += symbol_size

                symbol_count = len(buff) // symbol_size
                new_b += (symbol_count | 0x80).to_bytes(control_size, byteorder='big') + buff
                n = 1
    
    if i + symbol_size <= len(b):
        if n > 1:
            new_b += n.to_bytes(control_size, byteorder='big') + b[i:i+symbol_size]
        else:
            new_b += ((1 | 0x80).to_bytes(control_size, byteorder='big') + b[i:i+symbol_size])

    return new_b

def deRLE_step3(b, Ms = 8, Mc = 8):

    deB = b""
    i = 0
    symbol_size = Ms // 8
    control_size = Mc // 8
    while i < len(b):
        if b[i] & 0x80:
            count = b[i] & 0x7F
            deB += b[i+1:i+1+count*symbol_size]
            i += 1 + count * symbol_size
        else:
            deB += b[i+control_size:i+symbol_size+control_size]*b[i]
            i += 1 + symbol_size
    
    return deB

# Ms = 16
b = b"\xCF\xCE\xCF\xCE\xCF\xCE" 
# Ms = 24
#b = b"\xCF\xCE\xCF\xCF\xCE\xCF" 
print(b) # изначальная строка
new_b = RLE_step3(b)
print(new_b) # сжатая строка
deB = deRLE_step3(new_b)
print(deB) # разжатая строка
print('\n')

b'\xcf\xce\xcf\xce\xcf\xce'
b'\x86\xcf\xce\xcf\xce\xcf\xce'
b'\xcf\xce\xcf\xce\xcf\xce'




In [ ]:
def read_rle_file(filename):
    with open(filename, 'rb') as f:
        meta = f.read(2)
        Ms, Mc = meta[0], meta[1]
        data = f.read()
    return Ms, Mc, data

def write_rle_file(filename, data, Ms, Mc):
    with open(filename, 'wb') as f:
        f.write(bytes([Ms, Mc]))
        f.write(data)

def compress_file_rle(input_path, output_path, Ms=16, Mc=16):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    compressed = RLE_step3(data, Ms, Mc)
    write_rle_file(output_path, compressed, Ms, Mc)

def decompress_file_rle(input_path, output_path):
    Ms, Mc, compressed = read_rle_file(input_path)
    
    decompressed = deRLE_step3(compressed, Ms, Mc)
    
    with open(output_path, 'wb') as f:
        f.write(decompressed)

compress_file_rle("test2.txt", "test2_compression.txt")
decompress_file_rle("test2_compression.txt", "test2_compression_return.txt")

In [3]:
def get_utf8_char_size(byte):
    if byte & 0x80 == 0:
        return 1
    elif (byte & 0xE0) == 0xC0:
        return 2
    elif (byte & 0xF0) == 0xE0:
        return 3
    elif (byte & 0xF8) == 0xF0:
        return 4

def RLE_step4(b):
    #Ms = 16 # длина кода символа
    Mc = 8 # длина управляющего символа 

    new_b = b""
    n = 1
    i = 0
    while i < len(b):

        symbol_size1 = get_utf8_char_size(b[i])
        if i + symbol_size1 > len(b):
            break
        bytes1 = b[i:i+symbol_size1]

        if i + symbol_size1 >= len(b):
            break

        symbol_size2 = get_utf8_char_size(b[i+symbol_size1])
        if i + symbol_size1 + symbol_size2 > len(b):
            break
        bytes2 = b[i+symbol_size1:i+symbol_size1+symbol_size2]

        if bytes1 == bytes2:
            n += 1
            i += symbol_size1
        else:
            if n > 1:
                new_b += n.to_bytes(Mc//8, byteorder='big') + b[i:i+symbol_size1]
                i += symbol_size1
                n = 1
            else:
                buff = b""
                symbol_count = 0
                first_symbol_size = symbol_size1

                while i < len(b):
                    current_size = get_utf8_char_size(b[i])
                    if current_size != first_symbol_size:
                        break   

                    buff += b[i:i+current_size]
                    i += current_size
                    symbol_count += 1

                    if i >= len(b):
                        break
                    
                    symbol_size1 = get_utf8_char_size(b[i])
                    if i + symbol_size1 > len(b):
                        break
                    bytes1 = b[i:i+symbol_size1]

                    if i + symbol_size1 >= len(b):
                        break

                    symbol_size2 = get_utf8_char_size(b[i+symbol_size1])
                    if i + symbol_size1 + symbol_size2 > len(b):
                        break
                    bytes2 = b[i+symbol_size1:i+symbol_size1+symbol_size2]

                    if bytes1 == bytes2:
                        break
                
                new_b += (symbol_count | 0x80).to_bytes(Mc//8, byteorder='big') + buff
                n = 1
    
    if i < len(b):
        symbol_size = get_utf8_char_size(b[i])
        if n > 1:
            new_b += n.to_bytes(Mc//8, byteorder='big') + b[i:i+symbol_size]
        else:
            new_b += ((1 | 0x80).to_bytes(Mc//8, byteorder='big') + b[i:i+symbol_size])

    return new_b

def deRLE_step4(b):
    Ms = 16 # длина кода символа
    Mc = 8 # длина управляющего символа 

    deB = b""
    i = 0
    symbol_size = Ms // 8
    while i < len(b):
        if b[i] & 0x80:
            count = b[i] & 0x7F
            deB += b[i+1:i+1+count*symbol_size]
            i += 1 + count * symbol_size
        else:
            deB += b[i+(Mc//8):i+symbol_size+(Mc//8)]*b[i]
            i += 1 + symbol_size
    
    return deB

b = "АА  А ABABAB".encode('utf-8')
print(b)
print(RLE_step4(b))



b'\xd0\x90\xd0\x90  \xd0\x90 ABABAB'
b'\x02\xd0\x90\x02 \x81\xd0\x90\x86 ABABA\x81B'


In [ ]:
def read_rle_file(filename):
    with open(filename, 'rb') as f:
        meta = f.read(2)
        Ms, Mc = meta[0], meta[1]
        data = f.read()
    return Ms, Mc, data

def write_rle_file(filename, data, Ms, Mc):
    with open(filename, 'wb') as f:
        f.write(bytes([Ms, Mc]))
        f.write(data)

def compress_file_rle(input_path, output_path, Ms=16, Mc=8):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    compressed = RLE_step3(data)
    write_rle_file(output_path, compressed, Ms, Mc)
    
    print(f"Сжато: {len(data)} → {len(compressed) + 2} байт")

def decompress_file_rle(input_path, output_path):
    Ms, Mc, compressed = read_rle_file(input_path)
    
    decompressed = deRLE_step3(compressed, Ms, Mc)
    
    with open(output_path, 'wb') as f:
        f.write(decompressed)
    
    print(f"Распаковано: {len(decompressed)} байт")